# Goal or No Goal?

**Complete | 100pts**
*Instructions | Submissions | Resources*

A GHZ state is the many-qubit generalization of a Bell pair:

$$\vert{}GHZ_n\rangle = \frac{1}{\sqrt{2}}(\vert{}00\dots0\rangle + \vert{}11\dots1\rangle).$$

A natural next question is how you would prove you actually built one, on hardware, where all you ever get back are the expectation values of observables you chose to measure (and not the exact state, like you can do with simulators).

One obvious first attempt is to measure every qubit in the computational basis and look at the histogram of bitstrings. For the GHZ state you get only two of them, all-zeros and all-ones, each with probability $1/2$, and every other bitstring should never appear. But that is not sufficient.

Consider a machine that flips a fair coin and, depending on the result, prepares either $\vert{}00\dots0\rangle$ or $\vert{}11\dots1\rangle$. That state can be described by the matrix:

$$\rho_{mix} = \frac{1}{2}\vert{}00\dots0\rangle\langle00\dots0\vert{} + \frac{1}{2}\vert{}11\dots1\rangle\langle11\dots1\vert{}.$$

This machine produces exactly the same $Z$-basis sampling histogram as a GHZ state. No number of computational-basis shots will ever separate the two, because the difference between them lives entirely in the off-diagonal terms of the matrix (known as the coherences, which characterize the relationship between states) and $Z$ basis measurements are blind to these coherences.

Separating these two cases requires an observable that is sensitive to the off-diagonal terms (the coherences). The measure that sees the coherences is the global $X$-basis parity, described by:

$$\Pi_X = X_0 X_1 \cdots X_{n-1}.$$

A parity observable is a product of single-qubit Pauli operators, for example $Z_0Z_1$ or $X_0X_1X_2$. Because each Pauli squares to the identity, such a product has only two eigenvalues, $+1$ and $-1$. Measuring it therefore asks a single yes-or-no question rather than reading out individual qubits. Specifically, the question it asks is one of evenness. The global $X$-basis parity reports whether the number of qubits found in $\vert{}-\rangle$ is even or odd.

$$\langle GHZ_n \vert{} \Pi_X \vert{} GHZ_n \rangle = +1, \quad Tr(\rho_{mix} \Pi_X) = 0.$$

## One more catch

There is one more subtlety. The $X$ parity on its own also does not uniquely identify the GHZ state. Consider the completely unentangled product state $\vert{}+\rangle^{\otimes n}$. Every qubit is a $+1$ eigenstate of its own $X$, so

$$\langle +\vert{}^{\otimes n} \Pi_X \vert{}+\rangle^{\otimes n} = +1$$

for every $n$. This is a case where we have perfect $X$ parity, but no entanglement in the state so we could once again be led astray!

In other words, the $X$ parity is blind to the populations (the distribution of how often the system of qubits is in one state versus another) in exactly the way the $Z$ histogram is blind to the coherences (the measure of the relationship that exists between states).

Neither test is sufficient on its own, but together they are. We can write down a list of $n$ parity checks that the GHZ state passes and no other state does, as:

$$\Pi_X = X_0 X_1 \cdots X_{n-1}, \quad Z_i Z_{i+1} \text{ for } i=0, 1, \dots, n-2.$$

The $n-1$ neighbouring $ZZ$ parities pin the populations onto all-zeros and all-ones; the single global $X$ parity certifies that those two branches are coherently superposed.

So the quantity to report is the mean of all $n$ checks:

$$s_n = \frac{1}{n} \left( \langle \Pi_X \rangle + \sum_{i=0}^{n-2} \langle Z_i Z_{i+1} \rangle \right),$$

which equals $+1$ if and only if the state is $\vert{}GHZ_n\rangle$.

Other states that would have given us similar results of one of these measures do not satisfy this fuller test. For example:

* $\vert{}+\rangle^{\otimes n}$ scores $1/n$
* the classical mixture scores $(n-1)/n$
* the wrong-phase state $(\vert{}00\dots0\rangle - \vert{}11\dots1\rangle)/\sqrt{2}$ scores $(n-2)/n$

## Challenge code

Your task is to complete this function:

`prepare_and_build_checks`: prepares the state and returns the checks to measure on it. Its argument is:

* `n` (int): the number of qubits, guaranteed to be at least 2.

It returns:

* `checks` (list): the $n$ observables to measure. The order does not matter.

That function does two things: it applies the gates that prepare the GHZ state on the device wires, and it builds the list of $n$ observables to measure on that state. Everything around it is provided in the code provided for you and needs no changes. `wrapper(n)` creates the `n`-wire device and defines a QNode called `parities`, which calls your `prepare_and_build_checks(n)`, measures the expectation value of every observable in the list you return, and averages them into the score. Because your function is called from inside that QNode, every gate you apply there lands on the circuit, and every observable you return gets measured.

The grader inspects the observables you return, not just the number that comes out. It requires exactly $n$ of them, each a plain product of Pauli operators with coefficient $1$, and specifically the global $X$ parity together with the $n-1$ adjacent $Z_iZ_{i+1}$ pairs written above. Returning $n$ copies of the identity, or the same check $n$ times, scores nothing.

### Input

A single integer `n`, the number of qubits, for example `3`.

### Output

A float: the mean of the $n$ parity values, equal to `1.0` for a correctly prepared GHZ state. Your answer must match to within an absolute numerical tolerance of `1e-5`.

### Test cases

The following public test cases are available to you. Note that there are additional hidden test cases that we use to verify that your code is valid in full generality.

```text
test_input: 
n: 3
expected_output: 1.0

test_input: 
n: 5
expected_output: 1.0

```

The score is $1$ for every $n$, because it is an average of $n$ checks that a correct GHZ state passes perfectly. That makes the expected output uninformative on its own: what distinguishes a correct solution is which observables produced it, which is what the grader inspects. Floating-point arithmetic will leave you a little short of exactly $1$, which is why the tolerance is there.

Good luck!

In [ ]:
import json
import pennylane as qp
import pennylane.numpy as np

def wrapper(n: int) -> float:
    """Builds the n-qubit GHZ state and averages its n parity checks.

    Args:
        n (int): The number of qubits. Guaranteed to satisfy n >= 2.

    Returns:
        float: The mean of the n parity values -- the global X parity together with
            the n-1 neighbouring ZZ parities. This equals +1 if and only if the
            state is the GHZ state.
    """
    dev = qp.device("default.qubit", wires=n)

    @qp.qnode(dev)
    def parities():
        return [qp.expval(observable) for observable in prepare_and_build_checks(n)]

    return float(np.mean(np.array(parities())))


def prepare_and_build_checks(n):
    """Prepares the GHZ state and lists the parity checks to measure on it.

    This runs inside the QNode above, so any gate you apply here is applied to the
    circuit, and the observables you return are the ones that get measured.

    Args:
        n (int): the number of qubits, guaranteed to be at least 2.

    Returns:
        list[Operator]: the n observables to measure -- the global X parity and the
            n-1 neighbouring ZZ parities. The order does not matter.
    """
    # Step 1: build the GHZ state -- a Hadamard on wire 0, then a CNOT fan-out
    #         from wire 0 to every other wire.
    # Step 2: build the global X parity: an X on every wire, multiplied together
    #         into a single observable with qp.prod and assign it 'checks'.
    # Step 3: build the n-1 neighbouring ZZ parities, one for each adjacent pair,
    #         indexed by the left-hand qubit of the pair and add it to 'checks'.
    # Step 4: collect all n observables into checks and return it.


#Inicia el codigo Aqui

# Put your code here.

    checks = [qp.prod(*[qp.PauliX(wires=wire) for wire in range(n)])]

    for i in range(n - 1):
        checks.append(qp.prod(qp.PauliZ(wires=i), qp.PauliZ(wires=i + 1)))
    

    return checks
# Termina el Codigo Aqui

# These functions are responsible for testing the solution.
AUDIT = {"tapes": []}
_real_device = qp.device


def audited_device(*args, **kwargs):
    """Wraps a device so the grader can see which observables were measured."""
    dev = _real_device(*args, **kwargs)
    inner = dev.execute

    def execute(circuits, *args, **kwargs):
        batch = [circuits] if isinstance(circuits, qp.tape.QuantumScript) else circuits
        for tape in batch:
            AUDIT["tapes"].append(tape)
        return inner(circuits, *args, **kwargs)

    dev.execute = execute
    return dev


def pauli_key(observable):
    """Reduces an observable to a canonical Pauli-word string, or None."""
    try:
        sentence = qp.pauli.pauli_sentence(observable)
    except Exception:
        return None
    terms = list(sentence.items())
    if len(terms) != 1:
        return None
    word, coeff = terms[0]
    if not np.isclose(complex(coeff), 1.0):
        return None
    if len(word) == 0:
        return "identity"
    return "".join(str(w) + word[w] for w in sorted(word))


def expected_keys(n):
    """The canonical keys of the n checks the challenge asks for."""
    keys = ["".join(str(w) + "X" for w in range(n))]
    keys += [str(i) + "Z" + str(i + 1) + "Z" for i in range(n - 1)]
    return sorted(keys)


def violation(n):
    """Returns a message if the checks measured are not the ones asked for."""
    tapes = AUDIT["tapes"]
    if len(tapes) != 1:
        return ("Expected exactly one circuit execution, saw " + str(len(tapes))
                + ". All n checks are measured on the single QNode provided.")
    measurements = tapes[0].measurements
    for m in measurements:
        if type(m) is not qp.measurements.ExpectationMP:
            return ("Disallowed measurement: " + type(m).__name__ + ". Every check "
                    "must be the expectation value of an observable.")
    if len(measurements) != n:
        return ("You returned " + str(len(measurements)) + " checks for n = "
                + str(n) + ". There are exactly n of them: one global X parity "
                "and n-1 neighbouring ZZ parities.")
    keys = [pauli_key(m.obs) for m in measurements]
    if None in keys:
        return ("One of your checks is not a single Pauli word with coefficient 1. "
                "Each check is a plain product of Pauli operators.")
    if "identity" in keys:
        return ("One of your checks is the identity. Every state passes it, so it "
                "certifies nothing.")
    if sorted(keys) != expected_keys(n):
        return ("Your checks reduce to " + str(sorted(keys)) + " but the n asked "
                "for are " + str(expected_keys(n)) + ": an X on every wire, and a "
                "Z Z on each adjacent pair (i, i+1).")
    return None


NOT_GHZ = (
    "Not the GHZ state. A Z-basis histogram alone cannot see the X parity, and an "
    "X parity alone cannot see the Z-basis structure -- you need both."
)


def run(test_case_input: str) -> str:
    n = json.loads(test_case_input)
    AUDIT["tapes"] = []
    qp.device = audited_device
    try:
        value = wrapper(n)
    finally:
        qp.device = _real_device
    # Both verdicts are raised here rather than deferred to check(), so that they
    # cannot be bypassed by code that rebinds check() in the module namespace.
    message = violation(n)
    if message is not None:
        raise AssertionError(message)
    if not np.isclose(value, 1.0, atol=1e-5):
        raise AssertionError(NOT_GHZ)
    return json.dumps(value)


def check(solution_output: str, expected_output: str) -> None:
    sol = json.loads(solution_output)
    exp = json.loads(expected_output)
    assert np.isclose(sol, exp, atol=1e-5), NOT_GHZ


# These are the public test cases
test_cases = [
    ('3', '1.0'),
    ('5', '1.0')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")